In [ ]:
# Galaxy collision simulator - 1

import numpy as np
from scipy.ndimage import gaussian_filter
import subprocess, shutil, os
from IPython.display import Video, display

# -----------------------------
# Basic settings
# -----------------------------
WIDTH = 720
HEIGHT = 720
FPS = 24
OUTPUT = "galaxy_collision.mp4"


# Set False if you want a denser, slower render.
FAST_MODE = True

if FAST_MODE:
    N1, N2 = 3500, 3500
    N_STEPS = 750
    SAVE_EVERY = 5
else:
    N1, N2 = 9000, 9000
    N_STEPS = 1100
    SAVE_EVERY = 5

np.random.seed(11)

# -----------------------------
# Physics parameters
# -----------------------------
G = 1.0
DT = 0.05
SOFTEN = 0.18

M1 = 1.0
M2 = 1.0

DISK_SCALE = 2.2
N_ARMS = 2
ARM_PITCH = 0.55

INITIAL_SEP = 22.0
IMPACT_PARAM = 3.5
APPROACH_VEL = 0.48
INC2 = np.pi / 3.5

VIEW_START = 21
VIEW_END = 12


# -----------------------------
# Galaxy model
# -----------------------------
def make_galaxy(M_core, N, inclination=0.0):
    # Exponential disk: many stars near the centre, fewer far away
    r = np.random.exponential(DISK_SCALE, N)
    r = np.clip(r, 0.25, 12.5)

    # Simple spiral-arm pattern
    arm_idx = np.random.randint(0, N_ARMS, N)
    arm_offset = arm_idx * (2 * np.pi / N_ARMS)
    theta_spiral = np.log(r / 0.25) / ARM_PITCH + arm_offset
    spread = 0.28 + 0.22 * np.random.random(N)
    theta = theta_spiral + np.random.normal(0, spread)

    x = r * np.cos(theta)
    y = r * np.sin(theta)
    z = np.random.normal(0, 0.08, N)

    # Brightness: disk fading + random stellar luminosity
    base = np.exp(-r / 3.0)
    lum = np.random.lognormal(0, 0.7, N)
    brightness = base * lum * 2.0

    # A few very bright stars for cinematic sparkle
    n_super = max(1, int(N * 0.01))
    super_idx = np.random.choice(N, n_super, replace=False)
    brightness[super_idx] *= np.random.uniform(10, 25, n_super)

    # Rough colour temperature: warm stars dominate, a few hot stars
    temp = np.clip(0.55 - 0.04 * r + np.random.normal(0, 0.22, N), 0, 1)
    temp[super_idx] = np.clip(temp[super_idx] + 0.35, 0, 1)

    # Softened Kepler-like rotation curve
    v_circ = np.sqrt(G * M_core / np.sqrt(r**2 + 0.5))
    vx = -v_circ * np.sin(theta)
    vy =  v_circ * np.cos(theta)
    vz = np.zeros(N)

    pos = np.column_stack([x, y, z])
    vel = np.column_stack([vx, vy, vz])

    # Tilt the second galaxy so the collision looks more 3D
    if inclination != 0:
        c, s = np.cos(inclination), np.sin(inclination)
        R = np.array([[1, 0, 0],
                      [0, c, -s],
                      [0, s,  c]])
        pos = pos @ R.T
        vel = vel @ R.T

    return pos, vel, brightness, temp


def temp_to_rgb(t):
    r = np.ones_like(t)
    g = 0.32 + 0.66 * t**0.85
    b = np.where(
        t < 0.7,
        0.15 + 0.50 * (t / 0.7)**1.6,
        0.65 + 0.45 * ((t - 0.7) / 0.3)
    )
    return r, g, b


# -----------------------------
# Initial conditions
# -----------------------------
c1 = np.array([-INITIAL_SEP / 2, -IMPACT_PARAM / 2, 0.0])
c2 = np.array([ INITIAL_SEP / 2,  IMPACT_PARAM / 2, 0.0])

v1 = np.array([ APPROACH_VEL / 2, 0.0, 0.0])
v2 = np.array([-APPROACH_VEL / 2, 0.0, 0.0])

p1, vel1, b1, t1 = make_galaxy(M1, N1, inclination=0.0)
p2, vel2, b2, t2 = make_galaxy(M2, N2, inclination=INC2)

p1 += c1
p2 += c2
vel1 += v1
vel2 += v2

positions = np.vstack([p1, p2])
velocities = np.vstack([vel1, vel2])
brightness = np.concatenate([b1, b2])
temperature = np.concatenate([t1, t2])

r_mult, g_mult, b_mult = temp_to_rgb(temperature)


# -----------------------------
# Background stars
# -----------------------------
N_BG = 500
bg_pos = np.random.uniform(-25, 25, (N_BG, 2))
bg_bright = np.random.lognormal(-0.5, 0.8, N_BG) * 0.5
bg_temp = np.random.uniform(0.3, 1.0, N_BG)
bg_r, bg_g, bg_b = temp_to_rgb(bg_temp)


# -----------------------------
# Gravity
# Restricted N-body:
# particles feel both galaxy cores; cores feel each other.
# -----------------------------
def particle_accel(pos, c1, c2):
    r1 = c1 - pos
    d1 = (np.sum(r1**2, axis=1) + SOFTEN**2) ** 1.5
    a1 = G * M1 * r1 / d1[:, None]

    r2 = c2 - pos
    d2 = (np.sum(r2**2, axis=1) + SOFTEN**2) ** 1.5
    a2 = G * M2 * r2 / d2[:, None]

    return a1 + a2


def core_accel(c1, c2):
    r = c2 - c1
    d = (np.sum(r**2) + SOFTEN**2) ** 1.5
    return G * M2 * r / d, -G * M1 * r / d


# -----------------------------
# Run simulation
# -----------------------------
frames_pos = []
frames_c1 = []
frames_c2 = []

print("Running gravity simulation...")

for step in range(N_STEPS):
    # Leapfrog integration: stable enough for this kind of orbital motion
    a_p = particle_accel(positions, c1, c2)
    a_c1, a_c2 = core_accel(c1, c2)

    velocities += a_p * DT / 2
    v1 += a_c1 * DT / 2
    v2 += a_c2 * DT / 2

    positions += velocities * DT
    c1 += v1 * DT
    c2 += v2 * DT

    a_p = particle_accel(positions, c1, c2)
    a_c1, a_c2 = core_accel(c1, c2)

    velocities += a_p * DT / 2
    v1 += a_c1 * DT / 2
    v2 += a_c2 * DT / 2

    if step % SAVE_EVERY == 0:
        frames_pos.append(positions.copy())
        frames_c1.append(c1.copy())
        frames_c2.append(c2.copy())

    if step % 150 == 0:
        print(f"  step {step}/{N_STEPS}")

N_FRAMES = len(frames_pos)
print(f"Simulation frames: {N_FRAMES}")


# -----------------------------
# Renderer
# -----------------------------
yy, xx = np.mgrid[0:HEIGHT, 0:WIDTH].astype(np.float32)
cx_pix, cy_pix = WIDTH / 2, HEIGHT / 2

dist_norm = np.sqrt((xx - cx_pix)**2 + (yy - cy_pix)**2) / (WIDTH / 2)
vignette = np.clip(1.0 - 0.30 * dist_norm**2, 0, 1).astype(np.float32)

# Soft static nebula background
nebula = np.zeros((HEIGHT, WIDTH, 3), dtype=np.float32)
np.random.seed(99)

for _ in range(6):
    bx, by = np.random.uniform(0, WIDTH), np.random.uniform(0, HEIGHT)
    sigma = np.random.uniform(120, 280)
    intensity = np.random.uniform(0.012, 0.035)

    blob = intensity * np.exp(-((xx - bx)**2 + (yy - by)**2) / (2 * sigma**2))
    nebula[..., 0] += blob * 0.9
    nebula[..., 1] += blob * 0.5
    nebula[..., 2] += blob * 0.7

nebula = gaussian_filter(nebula, sigma=(8, 8, 0))


def splat(buf, pos2d, bright, rm, gm, bm, view_lim):
    # Sub-pixel additive splatting.

    px_per_unit = WIDTH / (2 * view_lim)

    x_px = (pos2d[:, 0] + view_lim) * px_per_unit
    y_px = (pos2d[:, 1] + view_lim) * px_per_unit

    mask = (x_px >= 0) & (x_px < WIDTH - 1) & (y_px >= 0) & (y_px < HEIGHT - 1)
    if not np.any(mask):
        return

    x = x_px[mask]
    y = y_px[mask]
    b = bright[mask]

    rm = rm[mask]
    gm = gm[mask]
    bm = bm[mask]

    xi = x.astype(np.int32)
    yi = y.astype(np.int32)

    fx = x - xi
    fy = y - yi

    w00 = (1 - fx) * (1 - fy) * b
    w10 = fx * (1 - fy) * b
    w01 = (1 - fx) * fy * b
    w11 = fx * fy * b

    for ch, m in enumerate((rm, gm, bm)):
        np.add.at(buf[:, :, ch], (yi, xi), w00 * m)
        np.add.at(buf[:, :, ch], (yi, xi + 1), w10 * m)
        np.add.at(buf[:, :, ch], (yi + 1, xi), w01 * m)
        np.add.at(buf[:, :, ch], (yi + 1, xi + 1), w11 * m)


def splat_nucleus(buf, core, view_lim, intensity=35.0):
    px_per_unit = WIDTH / (2 * view_lim)

    x = (core[0] + view_lim) * px_per_unit
    y = (core[1] + view_lim) * px_per_unit

    if not (0 <= x < WIDTH - 1 and 0 <= y < HEIGHT - 1):
        return

    xi = int(x)
    yi = int(y)
    fx = x - xi
    fy = y - yi

    colour = np.array([1.0, 0.90, 0.72])

    for ch in range(3):
        val = intensity * colour[ch]
        buf[yi, xi, ch] += (1 - fx) * (1 - fy) * val
        buf[yi, xi + 1, ch] += fx * (1 - fy) * val
        buf[yi + 1, xi, ch] += (1 - fx) * fy * val
        buf[yi + 1, xi + 1, ch] += fx * fy * val


trail_buf = np.zeros((HEIGHT, WIDTH, 3), dtype=np.float32)


def render_frame(idx):
    global trail_buf

    progress = idx / max(N_FRAMES - 1, 1)
    ease = progress * progress * (3 - 2 * progress)
    view_lim = VIEW_START + (VIEW_END - VIEW_START) * ease

    cur = np.zeros((HEIGHT, WIDTH, 3), dtype=np.float32)

    # Background field stars
    splat(
        cur,
        bg_pos,
        bg_bright * (1 - 0.55 * ease),
        bg_r,
        bg_g,
        bg_b,
        view_lim
    )

    # Galaxy stars
    p = frames_pos[idx]
    splat(cur, p[:, :2], brightness, r_mult, g_mult, b_mult, view_lim)

    # Bright cores
    core1 = frames_c1[idx]
    core2 = frames_c2[idx]
    sep = np.linalg.norm(core1 - core2)

    nuc_intensity = 22 + 38 * np.exp(-sep**2 / 16)
    splat_nucleus(cur, core1, view_lim, nuc_intensity)
    splat_nucleus(cur, core2, view_lim, nuc_intensity)

    # Motion trails
    trail_buf *= 0.80
    trail_buf += cur * 0.45

    base = cur * 1.4 + trail_buf * 0.65

    # Bloom
    bloom1 = gaussian_filter(base, sigma=(1.5, 1.5, 0))
    bloom2 = gaussian_filter(base, sigma=(5.0, 5.0, 0))
    bloom3 = gaussian_filter(base, sigma=(14.0, 14.0, 0))

    # Simple diffraction spikes from very bright points
    bright_mask = np.maximum(base - 5.5, 0)
    spike_h = gaussian_filter(bright_mask, sigma=(0.4, 24, 0))
    spike_v = gaussian_filter(bright_mask, sigma=(24, 0.4, 0))

    img = (
        base * 1.3
        + bloom1 * 1.2
        + bloom2 * 0.65
        + bloom3 * 0.28
        + (spike_h + spike_v) * 0.42
        + nebula
    )

    img *= vignette[..., None]


    x = img * 0.55
    a, b, c, d, e = 2.51, 0.03, 2.43, 0.59, 0.14
    tone = (x * (a * x + b)) / (x * (c * x + d) + e)

    # Slight warm grade
    tone[..., 0] *= 1.03
    tone[..., 2] *= 0.96
    tone = tone * 0.97 + 0.012

    return np.clip(tone * 255, 0, 255).astype(np.uint8)


# -----------------------------
# Encode video
# -----------------------------
if shutil.which("ffmpeg") is None:
    raise RuntimeError("ffmpeg not found. In Colab it should already be installed.")

print("Rendering video...")

cmd = [
    "ffmpeg", "-y",
    "-f", "rawvideo",
    "-vcodec", "rawvideo",
    "-s", f"{WIDTH}x{HEIGHT}",
    "-pix_fmt", "rgb24",
    "-r", str(FPS),
    "-i", "-",
    "-c:v", "libx264",
    "-pix_fmt", "yuv420p",
    "-crf", "18",
    "-preset", "medium",
    OUTPUT
]

proc = subprocess.Popen(cmd, stdin=subprocess.PIPE, stderr=subprocess.DEVNULL)

for i in range(N_FRAMES):
    frame = render_frame(i)
    proc.stdin.write(frame.tobytes())

    if i % 25 == 0:
        print(f"  frame {i}/{N_FRAMES}")

proc.stdin.close()
proc.wait()

print("Done:", OUTPUT)

display(Video(OUTPUT, embed=True))

Running gravity simulation...
  step 0/750
  step 150/750
  step 300/750
  step 450/750
  step 600/750
Simulation frames: 150
Rendering video...
  frame 0/150
  frame 25/150
  frame 50/150
  frame 75/150
  frame 100/150
  frame 125/150
Done: galaxy_collision.mp4
